# NileMini-8M-SiTU

Thin notebook for inspecting the canonical repository implementation. The source of truth lives in `src/nilemini/`; real cloud training is launched through `infra/modal_train.py`.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "nilemini").exists():
    raise RuntimeError("Run this notebook from the nilemini-rs repository root.")
sys.path.insert(0, str(ROOT / "src"))

from nilemini.config import MODEL, load_profile, load_sft_profile  # noqa: E402
from nilemini.generation import chat_prompt  # noqa: E402
from nilemini.tokenizer import load_tokenizer, tokenizer_sha256  # noqa: E402

print(MODEL)
print(f"parameters={MODEL.expected_parameter_count:,}")

## Frozen tokenizer and prompt contract


In [ ]:
tokenizer_path = ROOT / "artifacts" / MODEL.model_name / "tokenizer.json"
tokenizer = load_tokenizer(tokenizer_path)
hello_ids = chat_prompt(tokenizer, [{"role": "user", "content": "Hello"}])
assert hello_ids == [1, 4, 204, 45, 474, 84, 2, 204, 5, 204]
print("tokenizer sha256:", tokenizer_sha256(tokenizer_path))
print("Hello prompt:", hello_ids)

## Resolved run budgets


In [ ]:
for filename in ["smoke.json", "pilot_l4.json", "full_l4.json"]:
    profile = load_profile(ROOT / "configs" / "training" / filename)
    print(filename, f"{profile.train_tokens:,} train tokens", f"{profile.updates:,} updates")
sft = load_sft_profile(ROOT / "configs" / "training" / "sft_l4.json")
print("SFT:", f"{sft.train_examples:,} train + {sft.validation_examples:,} validation")

## Real training

The notebook intentionally does not duplicate the training loop. Use the repository CLI or Modal launcher:

```bash
uv run nilemini doctor
./scripts/modal_train.sh --stage smoke
./scripts/modal_train.sh --stage pilot
# after pilot review:
./scripts/modal_train.sh --stage full
./scripts/modal_train.sh --stage sft
./scripts/modal_train.sh --stage export
```
